In [74]:
import pandas as pd
from tensorflow import keras
import tensorflow as tf
from tensorflow.keras import layers, callbacks,utils

import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.preprocessing import StandardScaler


In [75]:
df=pd.read_csv("cleaned_dataset_taiwan_2months.csv")
df.head()

,date,sitename,county,aqi,status,so2,co,o3,pm10,pm2.5,no2,nox,no,siteid
0,2024-08-31 23:00:00,Hukou,Hsinchu County,62.0,Moderate,0.9,0.17,35.0,18.0,17.0,2.3,2.6,0.3,22
1,2024-08-31 23:00:00,Zhongming,Taichung City,50.0,Good,1.6,0.32,27.9,27.0,14.0,7.6,9.3,1.6,31
2,2024-08-31 23:00:00,Zhudong,Hsinchu County,45.0,Good,0.4,0.17,25.1,21.0,13.0,2.9,4.1,1.1,23
3,2024-08-31 23:00:00,Hsinchu,Hsinchu City,42.0,Good,0.8,0.20,30.0,19.0,10.0,4.0,4.8,0.7,24
4,2024-08-31 23:00:00,Toufen,Miaoli County,50.0,Good,1.0,0.16,33.5,18.0,14.0,1.8,3.1,1.2,25


In [76]:
features=['so2','co','o3','pm2.5','pm10','no2','nox','no']
status=['status']
X=df[features]
y=df[status]  


In [77]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
y_encoded = utils.to_categorical(y_encoded, num_classes=num_classes)

c:\Users\Administrator\anaconda3\Lib\site-packages\sklearn\preprocessing\_label.py:110: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


In [78]:
X_train,X_test,y_train,y_test=train_test_split(X,y_encoded,test_size=0.2,random_state=17)

In [79]:
scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

In [80]:
model = tf.keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    layers.Dense(32, activation='relu'),
    layers.Dropout(0.2), # Prevents overfitting
    layers.Dense(num_classes, activation='softmax') # Softmax for multi-class
])

c:\Users\Administrator\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [81]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=[
        'accuracy',
        tf.keras.metrics.Precision(name='precision'),
        tf.keras.metrics.Recall(name='recall')
    ]
)

In [82]:
my_callbacks = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath='best_model.keras', monitor='val_accuracy', save_best_only=True)
]

In [83]:
model.fit(
X_train, y_train,
epochs=50,
batch_size=32,
validation_split=0.2,
callbacks=my_callbacks,
verbose=1
)

Epoch 1/50
2519/2519 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step - accuracy: 0.9134 - loss: 0.2919 - precision: 0.9142 - recall: 0.9123 - val_accuracy: 0.9333 - val_loss: 0.1754 - val_precision: 0.9348 - val_recall: 0.9312
Epoch 2/50
2519/2519 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9301 - loss: 0.1803 - precision: 0.9311 - recall: 0.9289 - val_accuracy: 0.9360 - val_loss: 0.1574 - val_precision: 0.9366 - val_recall: 0.9352
Epoch 3/50
2519/2519 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9320 - loss: 0.1726 - precision: 0.9329 - recall: 0.9310 - val_accuracy: 0.9347 - val_loss: 0.1599 - val_precision: 0.9352 - val_recall: 0.9341
Epoch 4/50
2519/2519 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9341 - loss: 0.1683 - precision: 0.9351 - recall: 0.9327 - val_accuracy: 0.9376 - val_loss: 0.1543 - val_precision: 0.9383 - val_recall: 0.9368
Epoch 5/50
2519/2519 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step - accuracy: 0.9344 - loss: 0.1650 - precision: 0.9355 - recall: 0.9332 - val_accuracy: 0.9313 - va

In [ ]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

#  PREDICT
y_pred = rf_model.predict(X_test)

# CALCULATE GLOBAL METRICS
accuracy = accuracy_score(y_test, y_pred)
precision, recall, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='weighted')

print("--- Overall Categorical Performance (Global) ---")
print(f"Accuracy:  {accuracy:.2%}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")

# WHICH POLLUTANT AFFECTS AQI THE MOST
importances = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
print("\n--- Pollutant Importance Ranking ---")
print(importances)

--- Overall Categorical Performance (Global) ---
Accuracy:  93.72%
Precision: 0.9331
Recall:    0.9372
F1-Score:  0.9320

--- Pollutant Importance Ranking ---
pm2.5    0.314288
no2      0.120451
pm10     0.120120
o3       0.119835
nox      0.091015
co       0.085376
no       0.076202
so2      0.072714
dtype: float64


In [85]:
import joblib

# 1. Save the trained Random Forest model
joblib.dump(rf_model, 'rf_aqi_classifier.pkl')

# 2. Save the LabelEncoder (Crucial for decoding results later)
joblib.dump(le, 'aqi_label_encoder.pkl')

['aqi_label_encoder.pkl']